# Huy — A0/A1 trên Kaggle CPU

Notebook này **gọi trực tiếp các script trong snapshot của repo** (`scripts/run_a0.py`, tùy chọn `scripts/run_a1.py`); không chép hay viết lại logic inference. Tham số giữ như máy: Qwen2.5-0.5B-Instruct, pinned revision `7ae557604adf67be50417f59c2c2f167def9a775`, CPU FP32, 4 threads, retrieval budget 64, max new tokens 128, greedy decoding.

## Chuẩn bị Kaggle
1. Trong Notebook Settings, chọn **Accelerator: None** để bắt buộc chạy CPU và bật Internet để tải Qwen lần đầu. Kaggle sẽ dùng CPU/RAM của máy chủ; đây vẫn là đường chạy CPU trong code.
2. Đính kèm Dataset có snapshot project hiện tại. Notebook nhận được cả cấu trúc thư mục đã bung như `DATHofHuy/a0`, `DATHofHuy/data`, ... trong ảnh, hoặc file `kaggle_cpu_bundle.zip`. **Không cần clone GitHub**: remote repo hiện chưa chứa các thay đổi local mới nhất.
3. Để chạy A1, thêm `scripts/run_a1.py` và `permission/output/permissions_medium.jsonl`, `permission/output/roles_medium.json`. Không tải model weights lên Kaggle; notebook tự tải từ Hugging Face.

Cell tìm project root trong `/kaggle/input` (hoặc giải nén bundle nếu đã đính kèm dạng ZIP), xác thực hash snapshot, rồi gọi đúng script. Output được ghi ở `/kaggle/working/DATH-Agent-Graph/results/`. Cuối lượt tải thư mục results về máy vì Kaggle working storage là của phiên chạy.

In [ ]:
import hashlib
import importlib.util
import shutil
import sys
from zipfile import ZipFile
from pathlib import Path

# Kaggle images usually include CPU PyTorch. Install only packages that are missing;
# do not replace the existing PyTorch build.
missing = [name for name in ('torch', 'transformers', 'accelerate')
           if importlib.util.find_spec(name) is None]
if missing:
    print('Installing missing packages:', ', '.join(missing), flush=True)
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])

import torch
print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__)
print('Kaggle GPU visible:', torch.cuda.is_available(), '(project inference still explicitly uses CPU)')
print('Kaggle input:', Path('/kaggle/input').exists())

In [ ]:
INPUT_ROOT = Path('/kaggle/input')
WORK_ROOT = Path('/kaggle/working/DATH-Agent-Graph')
required = [
    Path('a0/generative_qa.py'),
    Path('evaluation/kg_index.py'),
    Path('evaluation/hits_at_1.py'),
    Path('scripts/run_a0.py'),
    Path('eval_set/eval_set.json'),
    Path('data/wikidata_big/kg/full.txt'),
]
bundles = list(INPUT_ROOT.rglob('kaggle_cpu_bundle.zip'))
if len(bundles) > 1:
    raise FileNotFoundError(f'Expected at most one kaggle_cpu_bundle.zip; found {bundles}')
search_root = INPUT_ROOT
if bundles:
    search_root = Path('/kaggle/working/kaggle_cpu_source')
    search_root.mkdir(parents=True, exist_ok=True)
    with ZipFile(bundles[0]) as archive:
        archive.extractall(search_root)
roots = [p.parent.parent for p in search_root.rglob('run_a0.py')
         if p.parent.name == 'scripts']
roots = [root for root in roots if all((root / rel).is_file() for rel in required)]
if len(roots) != 1:
    raise FileNotFoundError(
        f'Expected exactly one attached project snapshot containing {required}; found {roots}. '
        'Attach the project Dataset with a0/, evaluation/, scripts/, eval_set/ and data/ folders, or attach kaggle_cpu_bundle.zip.'
    )
SOURCE_ROOT = roots[0]

# Copy the same project source and only the A0 inputs. Kaggle input is read-only;
# outputs must go under /kaggle/working. Copy A1 files only when present.
WORK_ROOT.mkdir(parents=True, exist_ok=True)
for folder in ('a0', 'evaluation', 'scripts'):
    shutil.copytree(SOURCE_ROOT / folder, WORK_ROOT / folder, dirs_exist_ok=True,
                    ignore=shutil.ignore_patterns('__pycache__', '*.pyc'))
for relative in (
    Path('eval_set/eval_set.json'),
    Path('data/wikidata_big/kg/full.txt'),
    Path('permission/output/permissions_medium.jsonl'),
    Path('permission/output/roles_medium.json'),
):
    source = SOURCE_ROOT / relative
    if source.is_file():
        destination = WORK_ROOT / relative
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, destination)

for relative in required:
    assert (WORK_ROOT / relative).is_file(), f'Missing copied input: {relative}'
# Locked SHA256 manifest for this notebook's source snapshot. If any value fails,
# the Kaggle Dataset does not contain the same local code/data used to create it.
expected_hashes = {
    'a0/__init__.py': '102762c847ca0dacf771076cf6a5732b4a600de7ae1b3895407a918b2c80f1b8',
    'a0/cronkgqa_cpu.py': 'eec213fd3af279fa9709c846e2e195c9ed08a724e3b8a4d1268f91e9750b4c4f',
    'a0/generative_qa.py': '8ddd2ea99c1b6c6570a906f49ee1d06b08d71034ef279536a41af2677b9fafa3',
    'evaluation/__init__.py': '73fc30e815bcf5c9e4832e4769d41155936db3df4c430cd60c50493f90b50542',
    'evaluation/hits_at_1.py': 'fb30dbe3f5f98d33293f49f4c37fbe68cb04351235b12c00d6c7c9a20964a1c6',
    'evaluation/kg_index.py': '5f5202f2bc99339bfb5103b9309d734941f94fb2df9e98061ecc678c777eb76e',
    'evaluation/leakage.py': '25b9e72e8c7698dbd4cf7b81753aecc5647acd3d2a6d5a86517647c69dca3c36',
    'evaluation/metrics.py': 'ec4e7ef1458d6b8d4cdb9393159ebbb6bd0fce21a506d99a5b701922f3cc74d7',
    'evaluation/runner.py': '7d09cbb7e44251b3d6f718624c34d09ab79ee2e41171ed9536b0467626f53048',
    'evaluation/temporal_error.py': 'd92fd877aef6b7c20a31198c7472ecc6ceee583d87240d8901ee0bfc34c5f091',
    'scripts/audit_data_v2.py': 'd3a4fa91ce949715368a8c9268d8ecf2a90680fdc15eac1cd1e5dface10a6c0c',
    'scripts/evaluate_outputs.py': '8538e36e73d80ee6fa4647ae7bbb993a3bb2855a115f1f1c2bef4bc9da2914f6',
    'scripts/generate_permissions.py': '4b8cd60fcd19df6202b78214ce197a6e38a466808b29559cf7af741eb0bfc77c',
    'scripts/run_a0.py': 'cae957699bc369f135a91a637ad2bbcc60318f4a8670a27c45e3d6afd9bc79bc',
    'scripts/run_a1.py': 'd641cb960a9231e9d5bf40231d4d8139dd03c5f74442f2b51ae658d2ae9353c2',
    'scripts/select_eval_set.py': 'b6b552a582cae6fc81a2829da47b609d67bfd634ba750c3524964e9cd9306a99',
    'scripts/summarize_experiment.py': 'e77a6ecab46c5f70cbd66f825db4bfdbb3530fa9c3fcf436d8ee29deeb17d0a9',
    'scripts/verify_reproduce.py': '0a1d3b20740457a0d3d1fc17628bbac5fce467e0c28dc4b94e497d9e496c7655',
    'data/wikidata_big/kg/full.txt': '491a8b2ded719a65b74c54a2d2f5b9af12eaf29e924eeb33cf44ae5906b0eb60',
    'eval_set/eval_set.json': '1a03f06d963f84a6123e7aab834328770402fcc3c74a40ce966df6dc9987f498',
    'permission/output/permissions_medium.jsonl': '5ec1ea4576e1243ad7cb4052908ceeedf744aceb4ed69fbd77fccb88ac780bbd',
    'permission/output/roles_medium.json': '651d171ea030af4d6fe9cfba4ce5c3f69e0278a60a64e5a8510247076e759ba9',
}
for relative, expected in expected_hashes.items():
    path = WORK_ROOT / relative
    if not path.is_file():
        if relative.startswith('permission/'):
            continue  # permission files are required only for the optional A1 cell
        raise FileNotFoundError(f'Missing code/data from the locked snapshot: {relative}')
    actual = hashlib.sha256(path.read_bytes()).hexdigest()
    if actual != expected:
        raise ValueError(f'Snapshot mismatch for {relative}: expected {expected}, got {actual}')
print('Using project snapshot:', SOURCE_ROOT)
print('Working project:', WORK_ROOT)
print(f'Exact local code/data snapshot verified ({len(expected_hashes)} SHA256 checks).')
print('KG bytes:', (WORK_ROOT / 'data/wikidata_big/kg/full.txt').stat().st_size)
print('Eval set bytes:', (WORK_ROOT / 'eval_set/eval_set.json').stat().st_size)

## A0 — đúng lệnh máy local

Lệnh dưới đây giữ nguyên flags của local; `%run` thực thi nguyên file `scripts/run_a0.py` từ snapshot. Không bật GPU, permission hay as-of. Lần đầu cần Internet để tải model.

In [ ]:
%cd /kaggle/working/DATH-Agent-Graph
%run scripts/run_a0.py --threads 4 --retrieval-budget 64 --max-new-tokens 128

## A1 — tùy chọn

Chạy cell này nếu đã attach cả permission medium JSONL lẫn roles medium JSON. Nó gọi đúng script A1 của repo, cùng model/generation/retrieval flags như A0.

In [ ]:
a1_script = Path('scripts/run_a1.py')
permission_jsonl = Path('permission/output/permissions_medium.jsonl')
permission_roles = Path('permission/output/roles_medium.json')
if not (a1_script.is_file() and permission_jsonl.is_file() and permission_roles.is_file()):
    raise FileNotFoundError('Attach scripts/run_a1.py and both medium permission files before running A1.')
%run scripts/run_a1.py --tightness medium --threads 4 --retrieval-budget 64 --max-new-tokens 128

In [ ]:
import json
from pathlib import Path

a0_dir = Path('results/a0')
for name in ('outputs_A0.jsonl', 'metrics_A0.json', 'run_metadata.json'):
    path = a0_dir / name
    print(name, 'exists=', path.is_file(), 'bytes=', path.stat().st_size if path.is_file() else 0)
if (a0_dir / 'metrics_A0.json').is_file():
    print(json.dumps(json.loads((a0_dir / 'metrics_A0.json').read_text()), indent=2))
print('Outputs are saved in /kaggle/working/DATH-Agent-Graph/results/. Download that folder before the Kaggle session ends.')